In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_test.csv
/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_train.csv


In [12]:
data = np.loadtxt('/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_train.csv', delimiter=",", skiprows=1)
test_data = np.loadtxt('/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_test.csv',delimiter=",",skiprows=1)
np.random.shuffle(data)
split = int(0.8 * data.shape[0])
# Split training data into train + validation
X_train = data[:split, 1:].T / 255.0        # (784, 48000)
Y_train_labels = data[:split, 0].astype(int)
X_val = data[split:, 1:].T / 255.0          # (784, 12000)
Y_val_labels = data[split:, 0].astype(int)

# One-hot encode only the training labels → (10, m)
m = Y_train_labels.shape[0]
Y_train = np.zeros((10, m))
Y_train[Y_train_labels, np.arange(m)] = 1
print(f'Train shape: {data.shape}')
print(f'Test shape:  {test_data.shape}')

Train shape: (60000, 785)
Test shape:  (10000, 785)


In [13]:
def init_params():
    W1 = 0.1 * np.random.randn(128, 784)
    b1 = 0.1 * np.random.randn(128, 1)
    W2 = 0.1 * np.random.randn(10, 128)
    b2 = 0.1 * np.random.randn(10, 1)
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    return np.exp(Z) / np.sum(np.exp(Z), axis=0, keepdims=True)

In [14]:
def forward_pass(X, W1, b1, W2, b2):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

In [15]:
def back_prop(X, Y, W2, Z1, A1, Z2, A2, m):
    dZ2 = A2 - Y
    dW2 = dZ2.dot(A1.T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m
 
    dA1 = W2.T.dot(dZ2)
    dZ1 = dA1 * (Z1 > 0)
    dW1 = dZ1.dot(X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    return dW2, db2, dW1, db1

In [16]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - (alpha * dW1)
    b1 = b1 - (alpha * db1)
    W2 = W2 - (alpha * dW2)
    b2 = b2 - (alpha * db2)
    return W1, b1, W2, b2

In [17]:
def train(X, Y, X_val, Y_val_labels, epochs, alpha):
    W1, b1, W2, b2 = init_params()
    m = X.shape[1]

    for i in range(epochs):
        # Forward pass
        Z1, A1, Z2, A2 = forward_pass(X, W1, b1, W2, b2)

        # Backward pass
        dW2, db2, dW1, db1 = back_prop(X, Y, W2, Z1, A1, Z2, A2, m)

        # Update parameters
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

        # Print progress every 50 epochs
        if i % 50 == 0:
            loss = -np.mean(np.sum(Y * np.log(A2 + 1e-8), axis=0))
            val_acc = accuracy(X_val, Y_val_labels, W1, b1, W2, b2)
            print(f"Epoch {i:>4} | Loss: {loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    return W1, b1, W2, b2

In [18]:
def predict(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_pass(X, W1, b1, W2, b2)
    return np.argmax(A2, axis=0)

In [19]:
def accuracy(X, Y_labels, W1, b1, W2, b2):
    predictions = predict(X, W1, b1, W2, b2)
    return np.mean(predictions == Y_labels)

In [20]:
    # Train
W1, b1, W2, b2 = train(X_train, Y_train, X_val, Y_val_labels, epochs=500, alpha=0.1)

    # Evaluate training accuracy
acc = accuracy(X_train, Y_train_labels, W1, b1, W2, b2)
print(f"\nFinal Training Accuracy: {acc * 100:.2f}%")


    # Evaluate test accuracy
X_test = test_data[:,1:].T / 255.0
Y_test = test_data[:,0].astype(int)
test_acc = accuracy(X_test, Y_test, W1, b1, W2, b2)
print(f"Test Accuracy: {test_acc*100:.2f}%")

Epoch    0 | Loss: 2.4960 | Val Acc: 15.95%
Epoch   50 | Loss: 0.6751 | Val Acc: 82.79%
Epoch  100 | Loss: 0.4902 | Val Acc: 86.70%
Epoch  150 | Loss: 0.4202 | Val Acc: 88.34%
Epoch  200 | Loss: 0.3812 | Val Acc: 89.18%
Epoch  250 | Loss: 0.3553 | Val Acc: 89.92%
Epoch  300 | Loss: 0.3361 | Val Acc: 90.48%
Epoch  350 | Loss: 0.3207 | Val Acc: 90.84%
Epoch  400 | Loss: 0.3080 | Val Acc: 91.22%
Epoch  450 | Loss: 0.2970 | Val Acc: 91.53%

Final Training Accuracy: 91.76%
Test Accuracy: 92.07%
